# [bus_stop_matching] 버스정류장을 100m 격자 셀에 매칭 (반경 100m 이내)

## 기능
- 버스정류장별 유동인구+위치정보 CSV와 100m 격자 데이터를 로드한다.
- Haversine 공식을 사용하여 각 100m 격자 셀의 중심점과 모든 버스정류장 간의 거리를 계산한다.
- 100m 이내에 위치한 버스정류장의 유동인구를 해당 격자 셀에 합산한다.
- 매칭된 정류장 수와 정류장 명칭 목록도 함께 기록한다.
- 결과를 `grid_with_bus_population_2025.csv`로 저장한다.

## 입력 파일
- `../data_processed/processed/bus_pop/버스정류장별_유동인구_위치정보_평균_2025_06_08.csv`
- `../data_processed/processed/grid/seoul_grid_100m.csv`

## 출력 파일
- `../data_processed/processed/bus_pop/grid_with_bus_population_2025.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

## BallTree 공간 인덱스를 활용한 거리 계산 설정

In [2]:
# BallTree는 haversine 메트릭을 내장하고 있어 별도 함수가 불필요
# 검색 반경: 100m → 라디안 변환 (지구 반지름 6,371,000m 기준)
EARTH_RADIUS_M = 6_371_000
SEARCH_RADIUS_M = 100
SEARCH_RADIUS_RAD = SEARCH_RADIUS_M / EARTH_RADIUS_M

print(f"검색 반경: {SEARCH_RADIUS_M}m → {SEARCH_RADIUS_RAD:.10f} rad")

검색 반경: 100m → 0.0000156961 rad


## 데이터 로드

In [3]:
# 데이터 로드
bus_stops = pd.read_csv('../data_processed/processed/bus_pop/버스정류장별_유동인구_위치정보_평균_2025_06_08.csv')
grid_data = pd.read_csv('../data_processed/processed/grid/seoul_grid_100m.csv')

# 버스정류장에서 유효한 위도/경도가 있는 데이터만 필터링
bus_stops_valid = bus_stops.dropna(subset=['Latitude', 'Longitude']).copy()

print(f"버스정류장 수 (전체): {len(bus_stops)}")
print(f"버스정류장 수 (유효 좌표): {len(bus_stops_valid)}")
print(f"그리드 셀 수: {len(grid_data)}")

버스정류장 수 (전체): 12532
버스정류장 수 (유효 좌표): 10652
그리드 셀 수: 92398


In [4]:
bus_stops.head()

,표준버스정류장ID,유동인구_평균,정류소명,Longitude,Latitude
0,100000001,32746.00,종로2가사거리,126.987752,37.569806
1,100000002,136235.67,창경궁.서울대학교병원,126.996521,37.579433
2,100000003,198718.67,명륜3가.성대입구,126.998251,37.582580
3,100000004,54661.67,종로2가.삼일교,126.987613,37.568579
4,100000005,133375.33,혜화동로터리.여운형활동터,127.001744,37.586243


In [5]:
bus_stops.describe()

,표준버스정류장ID,유동인구_평균,Longitude,Latitude
count,1.253200e+04,12532.000000,10652.000000,10652.000000
mean,1.329676e+08,22781.987378,126.987280,37.550921
std,7.827599e+07,37038.120695,0.086306,0.055127
min,1.000000e+08,1.000000,126.797811,37.430520
25%,1.089001e+08,3529.085000,126.917576,37.502511
50%,1.150009e+08,9852.000000,126.996565,37.550659
75%,1.220001e+08,26244.252500,127.052399,37.591567
max,9.998000e+08,566519.670000,127.181734,37.690177


In [6]:
# 실제 값 직접 확인
print(bus_stops['Longitude'].iloc[0])  # → 126.9877522923
print(bus_stops['Latitude'].iloc[0])   # → 37.5698055407
pd.reset_option('display.float_format')

126.9877522923
37.5698055407


## 버스정류장 → 격자 셀 매칭 (반경 100m)

In [7]:
import time
start_time = time.time()

# ── 1) 좌표를 라디안으로 변환 (BallTree haversine 메트릭 요구사항) ──
bus_coords_rad = np.radians(bus_stops_valid[['Latitude', 'Longitude']].values)
grid_coords_rad = np.radians(grid_data[['center_lat', 'center_lon']].values)

# ── 2) 버스정류장 BallTree 구축 ──
tree = BallTree(bus_coords_rad, metric='haversine')
print(f"BallTree 구축 완료 (버스정류장 {len(bus_coords_rad):,}개)")

# ── 3) 모든 그리드 셀에 대해 반경 내 버스정류장 한 번에 조회 ──
indices_list = tree.query_radius(grid_coords_rad, r=SEARCH_RADIUS_RAD)
print(f"반경 검색 완료 (그리드 셀 {len(grid_coords_rad):,}개)")

# ── 4) 결과 집계 (NumPy 배열로 빠르게 처리) ──
bus_pop = bus_stops_valid['유동인구_평균'].values
bus_names = bus_stops_valid['정류소명'].astype(str).values

유동인구_합계 = np.zeros(len(grid_data))
매칭_수 = np.zeros(len(grid_data), dtype=int)
정류장_목록 = [''] * len(grid_data)

for i, matched_idx in enumerate(indices_list):
    n = len(matched_idx)
    if n > 0:
        유동인구_합계[i] = bus_pop[matched_idx].sum()
        매칭_수[i] = n
        정류장_목록[i] = '|'.join(bus_names[matched_idx])

grid_data['유동인구_합계'] = 유동인구_합계
grid_data['매칭_버스정류장_수'] = 매칭_수
grid_data['매칭_정류장_목록'] = 정류장_목록

elapsed = time.time() - start_time
print(f"\n매칭 완료! 소요 시간: {elapsed:.1f}초")

BallTree 구축 완료 (버스정류장 10,652개)
반경 검색 완료 (그리드 셀 92,398개)

매칭 완료! 소요 시간: 0.4초


## 결과 저장 및 요약 통계

In [ ]:
# 결과 저장
output_file = "../data_processed/processed/bus_pop/grid_with_bus_population_2025.csv"
grid_data.to_csv(output_file, index=False, encoding='utf-8-sig')

# 요약 통계 출력
matched_grids = grid_data[grid_data['매칭_버스정류장_수'] > 0]
print(f"=== 매칭 결과 요약 ===")
print(f"전체 그리드 셀 수: {len(grid_data)}")
print(f"버스정류장이 매칭된 그리드 셀 수: {len(matched_grids)}")
print(f"총 유동인구 합계: {grid_data['유동인구_합계'].sum():,.0f}")
print(f"매칭된 버스정류장 수 (중복 포함): {grid_data['매칭_버스정류장_수'].sum()}")
print(f"\n결과 파일 저장: {output_file}")

print("\n처리 완료!")

=== 매칭 결과 요약 ===
전체 그리드 셀 수: 92398
버스정류장이 매칭된 그리드 셀 수: 25950
총 유동인구 합계: 1,286,925,549
매칭된 버스정류장 수 (중복 포함): 49093

결과 파일 저장: ../data_processed/processed/bus_pop/grid_with_bus_population_2025.csv

처리 완료!


In [11]:
df = pd.read_csv('../data_processed/processed/bus_pop/grid_with_bus_population_2025.csv')
df_pop = df[["cell_id", "유동인구_합계"]]
pd.set_option('display.float_format', '{:,.0f}'.format)
grid_data.describe()

df_pop.describe()

,cell_id,유동인구_합계
count,"92,398","92,398"
mean,"1,413,823,164,475","13,928"
std,"991,390,366","46,288"
min,"1,411,415,045,163",0
25%,"1,413,085,045,004",0
50%,"1,413,915,045,264",0
75%,"1,414,605,045,120","3,516"
max,"1,415,945,045,137","1,591,332"


In [12]:
pd.reset_option('display.float_format')

In [13]:
df.head()

,system:index,min_lon,max_lat,center_lat,max_lon,center_lon,min_lat,cell_id,유동인구_합계,매칭_버스정류장_수,매칭_정류장_목록
0,+141141+45163,126.789118,37.552496,37.552140,126.790016,126.789567,37.551784,1411415045163,0.0,0,NaN
1,+141141+45164,126.789118,37.553208,37.552852,126.790016,126.789567,37.552496,1411415045164,0.0,0,NaN
2,+141141+45165,126.789118,37.553921,37.553564,126.790016,126.789567,37.553208,1411415045165,0.0,0,NaN
3,+141142+45158,126.790016,37.548935,37.548579,126.790914,126.790465,37.548223,1411425045158,0.0,0,NaN
4,+141142+45159,126.790016,37.549647,37.549291,126.790914,126.790465,37.548935,1411425045159,0.0,0,NaN
